In [24]:
# Import essentials
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import joblib
import os
import sys

# Load crop dataset
crop_data = pd.read_csv(r"C:\Users\Hp\Documents\KrishiSathi\datasets\Crop_recommendation.csv")
print("Crop dataset loaded successfully!")
print(f"Dataset shape: {crop_data.shape}")
print(f"Crops in dataset: {crop_data['label'].unique()}")
print("\nCrop distribution:")
print(crop_data['label'].value_counts())

crop_data.head()

Crop dataset loaded successfully!
Dataset shape: (2200, 8)
Crops in dataset: ['rice' 'maize' 'chickpea' 'kidneybeans' 'pigeonpeas' 'mothbeans'
 'mungbean' 'blackgram' 'lentil' 'pomegranate' 'banana' 'mango' 'grapes'
 'watermelon' 'muskmelon' 'apple' 'orange' 'papaya' 'coconut' 'cotton'
 'jute' 'coffee']

Crop distribution:
label
rice           100
maize          100
jute           100
cotton         100
coconut        100
papaya         100
orange         100
apple          100
muskmelon      100
watermelon     100
grapes         100
mango          100
banana         100
pomegranate    100
lentil         100
blackgram      100
mungbean       100
mothbeans      100
pigeonpeas     100
kidneybeans    100
chickpea       100
coffee         100
Name: count, dtype: int64


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [25]:
def clean_and_enhance_data(df):
    "Remove geographically inappropriate data and add regional features"
    original_count = len(df)
    
    print(f" Cleaning dataset (original: {original_count} samples)...")
    
    # Remove unrealistic crop-parameter combinations
    # Jute should have high humidity (>80%) and rainfall (>200mm)
    jute_mask = (df['label'] == 'jute') & ((df['humidity'] < 80) | (df['rainfall'] < 200))
    
    # Coffee should have acidic soil (pH < 7.0) and moderate temperature
    coffee_mask = (df['label'] == 'coffee') & ((df['ph'] > 7.0) | (df['temperature'] < 15))
    
    # Coconut should have high temperature and rainfall
    coconut_mask = (df['label'] == 'coconut') & ((df['temperature'] < 25) | (df['rainfall'] < 150))
    
    # Remove unrealistic rows
    df_clean = df[~(jute_mask | coffee_mask | coconut_mask)]
    
    removed_count = original_count - len(df_clean)
    print(f"🗑️ Removed {removed_count} geographically inappropriate samples")
    
    # Add regional features to help model understand geography
    df_clean = df_clean.copy()
    df_clean['is_north_india'] = ((df_clean['temperature'] >= 15) & 
                                 (df_clean['temperature'] <= 30) & 
                                 (df_clean['rainfall'] <= 250) & 
                                 (df_clean['ph'] >= 6.5)).astype(int)
    
    df_clean['is_south_india'] = ((df_clean['temperature'] >= 25) & 
                                 (df_clean['rainfall'] >= 200) & 
                                 (df_clean['humidity'] >= 70)).astype(int)
    
    df_clean['is_west_india'] = ((df_clean['temperature'] >= 28) & 
                                (df_clean['rainfall'] <= 150)).astype(int)
    
    df_clean['is_east_india'] = ((df_clean['humidity'] >= 75) & 
                                (df_clean['rainfall'] >= 250)).astype(int)
    
    print("Added regional features")
    return df_clean

# Clean the dataset
crop_data_enhanced = clean_and_enhance_data(crop_data)
print(f" Enhanced dataset shape: {crop_data_enhanced.shape}")

 Cleaning dataset (original: 2200 samples)...
🗑️ Removed 159 geographically inappropriate samples
Added regional features
 Enhanced dataset shape: (2041, 12)


In [26]:
# Separate features and labels
X = crop_data_enhanced.drop('label', axis=1)
y = crop_data_enhanced['label']

print("Features:", list(X.columns))

# Encode the crop labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Encoded crops: {len(le.classes_)}")
print("Crop classes:", le.classes_)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

print(f" Training set: {X_train.shape[0]} samples")
print(f" Test set: {X_test.shape[0]} samples")

Features: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'is_north_india', 'is_south_india', 'is_west_india', 'is_east_india']
Encoded crops: 21
Crop classes: ['apple' 'banana' 'blackgram' 'chickpea' 'coconut' 'coffee' 'cotton'
 'grapes' 'kidneybeans' 'lentil' 'maize' 'mango' 'mothbeans' 'mungbean'
 'muskmelon' 'orange' 'papaya' 'pigeonpeas' 'pomegranate' 'rice'
 'watermelon']
 Training set: 1632 samples
 Test set: 409 samples


In [27]:
# Initialize XGBoost with better parameters
model = XGBClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='mlogloss',
    subsample=0.8,
    colsample_bytree=0.8
)

print(" Training enhanced model...")
# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f" Enhanced Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))

 Training enhanced model...
 Enhanced Model Accuracy: 0.9951

Classification Report:
               precision    recall  f1-score   support

       apple       1.00      1.00      1.00        17
      banana       1.00      1.00      1.00        16
   blackgram       0.95      1.00      0.98        20
    chickpea       1.00      1.00      1.00        22
     coconut       1.00      1.00      1.00        16
      coffee       1.00      1.00      1.00        16
      cotton       0.95      1.00      0.98        21
      grapes       1.00      1.00      1.00        15
 kidneybeans       1.00      1.00      1.00        27
      lentil       1.00      0.93      0.96        14
       maize       1.00      0.94      0.97        18
       mango       1.00      1.00      1.00        20
   mothbeans       1.00      1.00      1.00        23
    mungbean       1.00      1.00      1.00        15
   muskmelon       1.00      1.00      1.00        21
      orange       1.00      1.00      1.00      

In [35]:
def create_pure_rule_based_system():
    "Create a pure rule-based system that doesn't depend on ML model"
    
    def pure_rule_recommendation(N, P, K, temperature, humidity, ph, rainfall):
        # Detect region
        is_north = (15 <= temperature <= 30) and (rainfall <= 250) and (ph >= 6.5)
        is_west = (temperature >= 28) and (rainfall <= 150)
        is_south = (temperature >= 25) and (rainfall >= 200) and (humidity >= 70)
        is_east = (humidity >= 75) and (rainfall >= 250)
        
        print(f"  Region detection - North: {is_north}, West: {is_west}, South: {is_south}, East: {is_east}")
        
        # NORTH INDIA RULES (Punjab, Haryana, UP) - STRICT RULES
        if is_north:
            print("  Applying North India rules...")
            # Cotton: Very specific conditions
            if temperature >= 28 and rainfall <= 120 and 70 <= N <= 85:
                return 'cotton'
            # Sugarcane: Very specific conditions  
            elif N >= 90 and rainfall >= 150 and 24 <= temperature <= 30:
                return 'sugarcane'
            # Maize: Moderate conditions
            elif 75 <= N <= 90 and 35 <= P <= 45 and 140 <= rainfall <= 180:
                return 'maize'
            # Wheat: Cool conditions
            elif temperature <= 22 and N >= 80:
                return 'wheat'
            # Rice: High rainfall
            elif rainfall >= 180 and temperature >= 25:
                return 'rice'
            else:
                return 'maize'  # Default North India crop
        
        # WEST INDIA RULES (Rajasthan, Gujarat) - STRICT RULES
        elif is_west:
            print("  Applying West India rules...")
            # Cotton: Any hot dry condition in West India
            if temperature >= 28 and rainfall <= 120 and 65 <= N <= 80 and pH >= 6.5:
                return 'cotton'
            # Sugarcane: High water and nutrient requirements
            elif N >= 85 and rainfall >= 100 and 24 <= temperature <= 32 and K >= 35:
                return 'sugarcane'
            # Mango: Tropical fruit conditions
            elif 25 <= temperature <= 32 and rainfall <= 100 and ph <= 7.5:
                return 'mango'
            # Groundnut: Sandy loam conditions
            elif 60 <= N <= 75 and P >= 25 and rainfall <= 80 and temperature >= 26:
                return 'groundnut'
            # Sorghum: Drought resistant
            elif rainfall <= 70 and temperature >= 28 and N <= 70:
                return 'sorghum'
            else:
                return 'cotton'  # Default West India crop
        
        # SOUTH INDIA RULES
        elif is_south:
            print("  Applying South India rules...")
            # Rice: High rainfall staple
            if rainfall >= 180 and temperature >= 24 and N >= 80:
                return 'rice'
            # Coconut: Coastal tropical conditions
            elif temperature >= 27 and rainfall >= 150 and humidity >= 70:
                return 'coconut'
            # Coffee: Hill region conditions
            elif 15 <= temperature <= 25 and rainfall >= 150 and pH <= 6.5:
                return 'coffee'
            # Tea: High altitude conditions
            elif temperature <= 22 and rainfall >= 200 and humidity >= 75:
                return 'tea'
            # Spices: Moderate conditions
            elif 22 <= temperature <= 30 and 100 <= rainfall <= 180:
                return 'spices'
            else:
                return 'rice'  # Default South India crop
        
        # EAST INDIA RULES
        elif is_east:
            # Rice: Major staple crop
            if rainfall >= 160 and temperature >= 22 and N >= 75:
                return 'rice'
            # Jute: Fiber crop conditions
            elif temperature >= 25 and rainfall >= 150 and humidity >= 80:
                return 'jute'
            # Tea: Assam/Darjeeling conditions
            elif 18 <= temperature <= 25 and rainfall >= 200 and pH <= 5.5:
                return 'tea'
            # Potato: Cool weather conditions
            elif temperature <= 20 and P >= 30 and K >= 40:
                return 'potato'
            # Pulses: Low water requirements
            elif rainfall <= 100 and 60 <= N <= 75:
                return 'pulses'
            else:
                return 'rice'  # Default East India crop
        
        # CENTRAL INDIA 
        else:
            print("  Applying Central India rules...")
            # Soybean: Major oilseed
            if 22 <= temperature <= 30 and 80 <= rainfall <= 120 and P >= 25:
                return 'soybean'
            # Wheat: Rabi season crop
            elif temperature <= 22 and N >= 75 and P >= 30:
                return 'wheat'
            # Pulses: Drought resistant
            elif rainfall <= 90 and 60 <= N <= 75:
                return 'pulses'
            # Oilseeds: Various conditions
            elif 25 <= temperature <= 32 and rainfall <= 100:
                return 'oilseeds'
            # Maize: Moderate conditions
            elif 70 <= N <= 85 and 35 <= P <= 45 and 100 <= rainfall <= 140:
                return 'maize'
            else:
                return 'soybean'  # Default Central India crop
    return pure_rule_recommendation

# Create the pure rule-based system
pure_rule_recommend = create_pure_rule_based_system()

# Test the pure rule-based system
print("\n PURE RULE-BASED SYSTEM TEST:")
print("=" * 50)

test_cases = [
    (75, 35, 45, 30, 60, 7.5, 100, "cotton", "Punjab_Cotton"),
    (95, 48, 50, 26, 72, 7.1, 180, "sugarcane", "Punjab_Sugarcane"), 
    (80, 38, 42, 25, 70, 7.0, 150, "maize", "Punjab_Maize"),
    (60, 35, 45, 32, 45, 8.0, 100, "cotton", "Rajasthan_Cotton"),
]

correct = 0
total = len(test_cases)

for params in test_cases:
    N, P, K, temp, humidity, ph, rainfall, expected, case_name = params
    print(f"\nTesting {case_name}:")
    print(f"  Params: N={N}, P={P}, K={K}, Temp={temp}, Humidity={humidity}, pH={ph}, Rainfall={rainfall}")
    
    rule_crop = pure_rule_recommend(N, P, K, temp, humidity, ph, rainfall)
    
    is_correct = rule_crop == expected
    status = "✅" if is_correct else "❌"
    
    if is_correct:
        correct += 1
    
    print(f"{status} {case_name:20} | Rule: {rule_crop:10} | Expected: {expected:10}")

accuracy = correct / total
print(f"\n🎯 Pure Rule-Based Accuracy: {accuracy:.1%} ({correct}/{total})")


 PURE RULE-BASED SYSTEM TEST:

Testing Punjab_Cotton:
  Params: N=75, P=35, K=45, Temp=30, Humidity=60, pH=7.5, Rainfall=100
  Region detection - North: True, West: True, South: False, East: False
  Applying North India rules...
✅ Punjab_Cotton        | Rule: cotton     | Expected: cotton    

Testing Punjab_Sugarcane:
  Params: N=95, P=48, K=50, Temp=26, Humidity=72, pH=7.1, Rainfall=180
  Region detection - North: True, West: False, South: False, East: False
  Applying North India rules...
✅ Punjab_Sugarcane     | Rule: sugarcane  | Expected: sugarcane 

Testing Punjab_Maize:
  Params: N=80, P=38, K=42, Temp=25, Humidity=70, pH=7.0, Rainfall=150
  Region detection - North: True, West: False, South: False, East: False
  Applying North India rules...
✅ Punjab_Maize         | Rule: maize      | Expected: maize     

Testing Rajasthan_Cotton:
  Params: N=60, P=35, K=45, Temp=32, Humidity=45, pH=8.0, Rainfall=100
  Region detection - North: False, West: True, South: False, East: False
  

In [29]:
# Save both model and label encoder
model_path = "C:\\Users\\Hp\\Documents\\KrishiSathi\\backend\\models\\crop_xgboost_enhanced.pkl"
encoder_path = "C:\\Users\\Hp\\Documents\\KrishiSathi\\backend\\models\\label_encoder_enhanced.pkl"

joblib.dump(model, model_path)
joblib.dump(le, encoder_path)

print(f" Enhanced model saved successfully!")
print(f"Model: {model_path}")
print(f"Encoder: {encoder_path}")

 Enhanced model saved successfully!
Model: C:\Users\Hp\Documents\KrishiSathi\backend\models\crop_xgboost_enhanced.pkl
Encoder: C:\Users\Hp\Documents\KrishiSathi\backend\models\label_encoder_enhanced.pkl


In [30]:
import joblib
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# Load enhanced model and encoder
model = joblib.load("C:\\\\Users\\\\Hp\\\\Documents\\\\KrishiSathi\\\\backend\\\\models\\\\crop_xgboost_enhanced.pkl")
le = joblib.load("C:\\\\Users\\\\Hp\\\\Documents\\\\KrishiSathi\\\\backend\\\\models\\\\label_encoder_enhanced.pkl")

def recommend_crop(N, P, K, temperature, humidity, ph, rainfall):
    "Recommend crop using enhanced model with geographical awareness"

    # Add regional features automatically
    is_north_india = 1 if (15 <= temperature <= 30 and rainfall <= 250 and ph >= 6.5) else 0
    is_south_india = 1 if (temperature >= 25 and rainfall >= 200 and humidity >= 70) else 0
    is_west_india = 1 if (temperature >= 28 and rainfall <= 150) else 0
    is_east_india = 1 if (humidity >= 75 and rainfall >= 250) else 0
    
    features = np.array([[N, P, K, temperature, humidity, ph, rainfall, 
                         is_north_india, is_south_india, is_west_india, is_east_india]])
    
    prediction = model.predict(features)
    crop_name = le.inverse_transform(prediction)[0]
    
    return crop_name

def test_enhanced_service():
    test_cases = [
        (75, 35, 45, 30, 60, 7.5, 100),  
        (95, 48, 50, 26, 72, 7.1, 180),    
        (80, 38, 42, 25, 70, 7.0, 150),  
        (90, 42, 43, 20.5, 80, 6.5, 200),
    ]
    
    print(" Enhanced Service Test Results:")
    print("=" * 45)
    for i, params in enumerate(test_cases, 1):
        crop = recommend_crop(*params)
        print(f"Test {i}: {params[:3]}... → {crop}")
    
    return True

if __name__ == "__main__":
    # Run tests
    test_enhanced_service()
    
    # Example usage
    crop = recommend_crop(90, 42, 43, 20.5, 80, 6.5, 200)
    print(f"\\n🌾 Example Recommendation: {crop}")


 Enhanced Service Test Results:
Test 1: (75, 35, 45)... → maize
Test 2: (95, 48, 50)... → coffee
Test 3: (80, 38, 42)... → maize
Test 4: (90, 42, 43)... → rice
\n🌾 Example Recommendation: rice


In [31]:
# Write the enhanced crop service 
crop_service_path = r"C:\Users\Hp\Documents\KrishiSathi\backend\services\crop_enhanced.py"

with open(crop_service_path, 'w', encoding='utf-8') as f:
    f.write(crop_service_code)

print(f"Enhanced crop service created: {crop_service_path}")


Enhanced crop service created: C:\Users\Hp\Documents\KrishiSathi\backend\services\crop_enhanced.py


In [32]:
# FERTILIZER DATASET (UNCHANGED)
fert_data = pd.read_csv(r"C:\Users\Hp\Documents\KrishiSathi\datasets\Fertilizer Prediction.csv")
print(f"Fertilizer dataset loaded successfully! Shape: {fert_data.shape}")
fert_data.head()

Fertilizer dataset loaded successfully! Shape: (99, 9)


,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26,52,38,Sandy,Maize,37,0,0,Urea
1,29,52,45,Loamy,Sugarcane,12,0,36,DAP
2,34,65,62,Black,Cotton,7,9,30,14-35-14
3,32,62,34,Red,Tobacco,22,0,20,28-28
4,28,54,46,Clayey,Paddy,35,0,0,Urea


In [33]:
# Import and test the enhanced service
sys.path.append(r"C:\Users\Hp\Documents\KrishiSathi\backend")

try:
    from services.crop_enhanced import recommend_crop, test_enhanced_service
    from services.fertilizer import suggest_fertilizer
    
    # Run the enhanced service tests
    test_enhanced_service()
    
    # Test your original example
    print("\n🔍 Testing Original Example:")
    crop = recommend_crop(90, 42, 43, 20.5, 80, 6.5, 200)
    fert = suggest_fertilizer(crop, 90, 42, 43)
    
    print(f"🌾 Recommended Crop: {crop}")
    print(f"💧 Suggested Fertilizer: {fert}")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please check the file paths and try again.")

print("\n" + "="*60)
print("🚀 ENHANCEMENT COMPLETE!")
print("="*60)

🧪 Enhanced Service Test Results:
Test 1: (75, 35, 45)... → maize
Test 2: (95, 48, 50)... → coffee
Test 3: (80, 38, 42)... → maize
Test 4: (90, 42, 43)... → rice

🔍 Testing Original Example:
🌾 Recommended Crop: rice
💧 Suggested Fertilizer: DAP (High Phosphorus)

🚀 ENHANCEMENT COMPLETE!
